In [1]:
import json
import os
from zipfile import ZipFile

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix

In [2]:
kaggle_credentials = json.load(open('/content/kaggle.json'))
os.environ['KAGGLE_USERNAME'] = kaggle_credentials['username']
os.environ['KAGGLE_KEY'] = kaggle_credentials['key']

In [3]:
!kaggle datasets download masoudnickparvar/brain-tumor-mri-dataset


Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
brain-tumor-mri-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
with ZipFile('brain-tumor-mri-dataset.zip', 'r') as zip:
    zip.extractall()

In [5]:
import shutil
import os

# create combined directory
combined_dir = "/content/Combined"
os.makedirs(combined_dir, exist_ok=True)

classes = ["glioma", "meningioma", "notumor", "pituitary"]

for cls in classes:
    os.makedirs(os.path.join(combined_dir, cls), exist_ok=True)

    # copy from Training
    train_cls_path = f"/content/Training/{cls}"
    for img in os.listdir(train_cls_path):
        src = os.path.join(train_cls_path, img)
        dst = os.path.join(combined_dir, cls, f"train_{img}")
        shutil.copy(src, dst)

    # copy from Testing
    test_cls_path = f"/content/Testing/{cls}"
    for img in os.listdir(test_cls_path):
        src = os.path.join(test_cls_path, img)
        dst = os.path.join(combined_dir, cls, f"test_{img}")
        shutil.copy(src, dst)

    total = len(os.listdir(os.path.join(combined_dir, cls)))
    print(f"{cls}: {total} images")

glioma: 1800 images
meningioma: 1800 images
notumor: 1800 images
pituitary: 1800 images


In [6]:
base_dir = "/content"

## data generator

In [7]:
combined_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_generator = combined_datagen.flow_from_directory(
    combined_dir,
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    combined_dir,
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    subset='validation',
    shuffle=False
)

Found 5760 images belonging to 4 classes.
Found 1440 images belonging to 4 classes.


#model with frozen base(phase 1 )

In [8]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [9]:
## freeze all base layers
for layer in base_model.layers:
    layer.trainable = False

In [10]:
# custom head with dropout to prevent overfitting
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
## callbacks

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_model_phase1.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [12]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks_phase1
)

Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 517ms/step - accuracy: 0.4670 - loss: 1.1806
Epoch 1: val_accuracy improved from None to 0.68542, saving model to best_model_phase1.keras

Epoch 1: finished saving model to best_model_phase1.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 137s 594ms/step - accuracy: 0.5856 - loss: 0.9912 - val_accuracy: 0.6854 - val_loss: 0.7512
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 457ms/step - accuracy: 0.7392 - loss: 0.6758
Epoch 2: val_accuracy improved from 0.68542 to 0.72083, saving model to best_model_phase1.keras

Epoch 2: finished saving model to best_model_phase1.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 87s 481ms/step - accuracy: 0.7561 - loss: 0.6337 - val_accuracy: 0.7208 - val_loss: 0.7251
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step - accuracy: 0.7952 - loss: 0.5562
Epoch 3: val_accuracy improved from 0.72083 to 0.76458, saving model to best_model_phase1.keras

Epoch 3: finished saving model to best_model_phase1.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 85s 

##  fine tuning (phase 2)

In [13]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

for layer in base_model.layers[-40:]:
    layer.trainable = True

In [14]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
callbacks_phase2=[
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_model_phase2.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [16]:
history_fine=model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks_phase2
)

Epoch 1/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step - accuracy: 0.8023 - loss: 0.5635
Epoch 1: val_accuracy improved from None to 0.83611, saving model to best_model_phase2.keras

Epoch 1: finished saving model to best_model_phase2.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 125s 515ms/step - accuracy: 0.8111 - loss: 0.5325 - val_accuracy: 0.8361 - val_loss: 0.4739
Epoch 2/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 457ms/step - accuracy: 0.8332 - loss: 0.4527
Epoch 2: val_accuracy did not improve from 0.83611
180/180 ━━━━━━━━━━━━━━━━━━━━ 85s 472ms/step - accuracy: 0.8427 - loss: 0.4398 - val_accuracy: 0.8264 - val_loss: 0.4833
Epoch 3/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step - accuracy: 0.8569 - loss: 0.4102
Epoch 3: val_accuracy did not improve from 0.83611
180/180 ━━━━━━━━━━━━━━━━━━━━ 86s 476ms/step - accuracy: 0.8594 - loss: 0.3948 - val_accuracy: 0.8333 - val_loss: 0.4691
Epoch 4/10
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step - accuracy: 0.8610 - loss: 0.3786
Epoch 4: val_accuracy improved fr

##phase 3 for balacing the class weights

In [20]:
class_weights = {
    0: 3,
    1: 2,
    2: 1.0,
    3: 1.0
}

In [21]:
callbacks_phase3 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_model_final.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [22]:

history_weighted = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    class_weight=class_weights,
    callbacks=callbacks_phase3
)

Epoch 1/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 453ms/step - accuracy: 0.8984 - loss: 0.5199
Epoch 1: val_accuracy improved from None to 0.89028, saving model to best_model_final.keras

Epoch 1: finished saving model to best_model_final.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 116s 481ms/step - accuracy: 0.8967 - loss: 0.5213 - val_accuracy: 0.8903 - val_loss: 0.3151
Epoch 2/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 500ms/step - accuracy: 0.8930 - loss: 0.4974
Epoch 2: val_accuracy improved from 0.89028 to 0.89167, saving model to best_model_final.keras

Epoch 2: finished saving model to best_model_final.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 94s 521ms/step - accuracy: 0.8974 - loss: 0.4891 - val_accuracy: 0.8917 - val_loss: 0.3101
Epoch 3/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 465ms/step - accuracy: 0.9049 - loss: 0.4640
Epoch 3: val_accuracy improved from 0.89167 to 0.89653, saving model to best_model_final.keras

Epoch 3: finished saving model to best_model_final.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 88s 487ms/ste

In [24]:
# FINAL EVALUATION — only on val_generator now
y_pred = model.predict(val_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
class_names = list(val_generator.class_indices.keys())

print("=== Final Evaluation ===")
print(classification_report(val_generator.classes, y_pred_classes, target_names=class_names))
print("Confusion Matrix:")
print(confusion_matrix(val_generator.classes, y_pred_classes))

# threshold sweep for glioma recall
from sklearn.metrics import recall_score

best_threshold = 0.5
best_recall = 0

for threshold in np.arange(0.1, 0.55, 0.05):
    preds = []
    for pred in y_pred:
        if pred[0] >= threshold:
            preds.append(0)
        else:
            preds.append(np.argmax(pred))
    recall = recall_score(val_generator.classes, preds, average=None)[0]
    print(f"Threshold: {threshold:.2f} → Glioma Recall: {recall:.2f}")
    if recall > best_recall:
        best_recall = recall
        best_threshold = threshold

print(f"\nBest threshold: {best_threshold:.2f} → Glioma Recall: {best_recall:.2f}")

45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step
=== Final Evaluation ===
              precision    recall  f1-score   support

      glioma       0.93      0.79      0.85       360
  meningioma       0.84      0.86      0.85       360
     notumor       0.93      0.99      0.96       360
   pituitary       0.92      0.99      0.95       360

    accuracy                           0.91      1440
   macro avg       0.91      0.91      0.90      1440
weighted avg       0.91      0.91      0.90      1440

Confusion Matrix:
[[285  52  18   5]
 [ 18 310   8  24]
 [  2   3 355   0]
 [  3   2   0 355]]
Threshold: 0.10 → Glioma Recall: 0.89
Threshold: 0.15 → Glioma Recall: 0.86
Threshold: 0.20 → Glioma Recall: 0.86
Threshold: 0.25 → Glioma Recall: 0.83
Threshold: 0.30 → Glioma Recall: 0.83
Threshold: 0.35 → Glioma Recall: 0.83
Threshold: 0.40 → Glioma Recall: 0.81
Threshold: 0.45 → Glioma Recall: 0.80
Threshold: 0.50 → Glioma Recall: 0.79

Best threshold: 0.10 → Glioma Recall: 0.89


In [25]:
# final predictions with best threshold = 0.1
y_pred = model.predict(val_generator)

y_final = []
for pred in y_pred:
    if pred[0] >= 0.1:
        y_final.append(0)
    else:
        y_final.append(np.argmax(pred))

y_final = np.array(y_final)

print("=== Final Report (Threshold = 0.1) ===")
print(classification_report(val_generator.classes, y_final, target_names=class_names))
print(confusion_matrix(val_generator.classes, y_final))

# save model
model.save("brain_tumor_final.keras")
print("Model saved.")

45/45 ━━━━━━━━━━━━━━━━━━━━ 6s 123ms/step
=== Final Report (Threshold = 0.1) ===
              precision    recall  f1-score   support

      glioma       0.74      0.89      0.81       360
  meningioma       0.89      0.64      0.74       360
     notumor       0.94      0.98      0.96       360
   pituitary       0.93      0.97      0.95       360

    accuracy                           0.87      1440
   macro avg       0.88      0.87      0.87      1440
weighted avg       0.88      0.87      0.87      1440

[[320  25  14   1]
 [ 99 229   8  24]
 [  6   2 352   0]
 [  9   0   0 351]]
Model saved.


In [26]:
from sklearn.metrics import recall_score

y_pred = model.predict(val_generator)

print(f"{'Threshold':<12} {'Glioma Recall':<16} {'Meningioma Recall':<20} {'Accuracy'}")
print("-" * 60)

for threshold in np.arange(0.10, 0.55, 0.05):
    preds = []
    for pred in y_pred:
        if pred[0] >= threshold:
            preds.append(0)
        else:
            preds.append(np.argmax(pred))
    preds = np.array(preds)

    recalls = recall_score(val_generator.classes, preds, average=None)
    acc = np.mean(preds == val_generator.classes)

    print(f"{threshold:<12.2f} {recalls[0]:<16.2f} {recalls[1]:<20.2f} {acc:.2f}")

45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step
Threshold    Glioma Recall    Meningioma Recall    Accuracy
------------------------------------------------------------
0.10         0.89             0.64                 0.87
0.15         0.86             0.71                 0.88
0.20         0.86             0.76                 0.89
0.25         0.83             0.79                 0.90
0.30         0.83             0.82                 0.90
0.35         0.83             0.84                 0.91
0.40         0.81             0.85                 0.91
0.45         0.80             0.86                 0.91
0.50         0.79             0.86                 0.91


In [27]:
GLIOMA_THRESHOLD = 0.30

y_pred = model.predict(val_generator)

y_final = []
for pred in y_pred:
    if pred[0] >= GLIOMA_THRESHOLD:
        y_final.append(0)
    else:
        y_final.append(np.argmax(pred))

y_final = np.array(y_final)

print("=== FINAL MODEL (Threshold = 0.30) ===")
print(classification_report(val_generator.classes, y_final, target_names=class_names))
print(confusion_matrix(val_generator.classes, y_final))

45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 102ms/step
=== FINAL MODEL (Threshold = 0.30) ===
              precision    recall  f1-score   support

      glioma       0.88      0.83      0.86       360
  meningioma       0.87      0.82      0.84       360
     notumor       0.94      0.99      0.96       360
   pituitary       0.93      0.98      0.95       360

    accuracy                           0.90      1440
   macro avg       0.90      0.90      0.90      1440
weighted avg       0.90      0.90      0.90      1440

[[299  42  15   4]
 [ 33 295   8  24]
 [  3   2 355   0]
 [  4   2   0 354]]
